In [11]:
import os, warnings, sys
from re import T

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

from abc import ABC, abstractmethod
import numpy as np
import joblib
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (
    Conv1D,
    Flatten,
    Dense,
    Conv1DTranspose,
    Reshape,
    Input,
    Layer,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Mean
from tensorflow.keras.backend import random_normal
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ============================================================================
# BASE CLASS
# ============================================================================

class Sampling(Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


class BaseVariationalAutoencoder(Model, ABC):
    model_name = None

    def __init__(
        self,
        seq_len,
        feat_dim,
        latent_dim,
        reconstruction_wt=3.0,
        batch_size=16,
        **kwargs,
    ):
        super(BaseVariationalAutoencoder, self).__init__(**kwargs)
        self.seq_len = seq_len
        self.feat_dim = feat_dim
        self.latent_dim = latent_dim
        self.reconstruction_wt = reconstruction_wt
        self.batch_size = batch_size
        self.total_loss_tracker = Mean(name="total_loss")
        self.reconstruction_loss_tracker = Mean(name="reconstruction_loss")
        self.kl_loss_tracker = Mean(name="kl_loss")
        self.encoder = None
        self.decoder = None

    def fit_on_data(self, train_data, max_epochs=1000, verbose=0):
        loss_to_monitor = "loss"
        early_stopping = EarlyStopping(
            monitor=loss_to_monitor, min_delta=1e-2, patience=50, mode="min"
        )
        reduce_lr = ReduceLROnPlateau(
            monitor=loss_to_monitor, factor=0.5, patience=30, mode="min"
        )
        self.fit(
            train_data,
            epochs=max_epochs,
            batch_size=self.batch_size,
            callbacks=[early_stopping, reduce_lr],
            verbose=verbose,
        )

    def call(self, X):
        z_mean, _, _ = self.encoder(X)
        x_decoded = self.decoder(z_mean)
        if len(x_decoded.shape) == 1:
            x_decoded = x_decoded.reshape((1, -1))
        return x_decoded

    def get_num_trainable_variables(self):
        trainableParams = int(
            np.sum([np.prod(v.get_shape()) for v in self.trainable_weights])
        )
        nonTrainableParams = int(
            np.sum([np.prod(v.get_shape()) for v in self.non_trainable_weights])
        )
        totalParams = trainableParams + nonTrainableParams
        return trainableParams, nonTrainableParams, totalParams

    def get_prior_samples(self, num_samples):
        Z = np.random.randn(num_samples, self.latent_dim)
        samples = self.decoder.predict(Z, verbose=0)
        return samples

    def get_prior_samples_given_Z(self, Z):
        samples = self.decoder.predict(Z)
        return samples

    @abstractmethod
    def _get_encoder(self, **kwargs):
        raise NotImplementedError

    @abstractmethod
    def _get_decoder(self, **kwargs):
        raise NotImplementedError

    def summary(self):
        self.encoder.summary()
        self.decoder.summary()

    def _get_reconstruction_loss(self, X, X_recons):
        def get_reconst_loss_by_axis(X, X_c, axis):
            x_r = tf.reduce_mean(X, axis=axis)
            x_c_r = tf.reduce_mean(X_recons, axis=axis)
            err = tf.math.squared_difference(x_r, x_c_r)
            loss = tf.reduce_sum(err)
            return loss

        # overall
        err = tf.math.squared_difference(X, X_recons)
        reconst_loss = tf.reduce_sum(err)

        reconst_loss += get_reconst_loss_by_axis(X, X_recons, axis=[2])  # by time axis
        return reconst_loss

    def train_step(self, X):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(X)

            reconstruction = self.decoder(z)

            reconstruction_loss = self._get_reconstruction_loss(X, reconstruction)

            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_sum(tf.reduce_sum(kl_loss, axis=1))

            total_loss = self.reconstruction_wt * reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)

        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, X):
        z_mean, z_log_var, z = self.encoder(X)
        reconstruction = self.decoder(z)
        reconstruction_loss = self._get_reconstruction_loss(X, reconstruction)

        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_sum(tf.reduce_sum(kl_loss, axis=1))

        total_loss = self.reconstruction_wt * reconstruction_loss + kl_loss

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def save_weights(self, model_dir):
        if self.model_name is None:
            raise ValueError("Model name not set.")
        encoder_wts = self.encoder.get_weights()
        decoder_wts = self.decoder.get_weights()
        joblib.dump(
            encoder_wts, os.path.join(model_dir, f"{self.model_name}_encoder_wts.h5")
        )
        joblib.dump(
            decoder_wts, os.path.join(model_dir, f"{self.model_name}_decoder_wts.h5")
        )

    def load_weights(self, model_dir):
        encoder_wts = joblib.load(
            os.path.join(model_dir, f"{self.model_name}_encoder_wts.h5")
        )
        decoder_wts = joblib.load(
            os.path.join(model_dir, f"{self.model_name}_decoder_wts.h5")
        )

        self.encoder.set_weights(encoder_wts)
        self.decoder.set_weights(decoder_wts)

    def save(self, model_dir):
        os.makedirs(model_dir, exist_ok=True)
        self.save_weights(model_dir)
        dict_params = {
            "seq_len": self.seq_len,
            "feat_dim": self.feat_dim,
            "latent_dim": self.latent_dim,
            "reconstruction_wt": self.reconstruction_wt,
            "hidden_layer_sizes": list(self.hidden_layer_sizes),
        }
        params_file = os.path.join(model_dir, f"{self.model_name}_parameters.pkl")
        joblib.dump(dict_params, params_file)


# ============================================================================
# TIMEVAE HELPER LAYERS
# ============================================================================

class TrendLayer(Layer):
    def __init__(self, feat_dim, trend_poly, seq_len, **kwargs):
        super(TrendLayer, self).__init__(**kwargs)
        self.feat_dim = feat_dim
        self.trend_poly = trend_poly
        self.seq_len = seq_len
        self.trend_dense1 = Dense(
            self.feat_dim * self.trend_poly, activation="relu", name="trend_params"
        )
        self.trend_dense2 = Dense(self.feat_dim * self.trend_poly, name="trend_params2")
        self.reshape_layer = Reshape(target_shape=(self.feat_dim, self.trend_poly))

    def call(self, z):
        trend_params = self.trend_dense1(z)
        trend_params = self.trend_dense2(trend_params)
        trend_params = self.reshape_layer(trend_params)  # shape: N x D x P

        lin_space = (
            tf.range(0, float(self.seq_len), 1) / self.seq_len
        )
        poly_space = tf.stack(
            [lin_space ** float(p + 1) for p in range(self.trend_poly)], axis=0
        )  # shape: P x T

        trend_vals = tf.matmul(trend_params, poly_space)  # shape (N, D, T)
        trend_vals = tf.transpose(trend_vals, perm=[0, 2, 1])  # shape: (N, T, D)
        trend_vals = tf.cast(trend_vals, tf.float32)

        return trend_vals


class SeasonalLayer(Layer):
    def __init__(self, feat_dim, seq_len, custom_seas, **kwargs):
        super(SeasonalLayer, self).__init__(**kwargs)
        self.feat_dim = feat_dim
        self.seq_len = seq_len
        self.custom_seas = custom_seas
        self.dense_layers = [
            Dense(feat_dim * num_seasons, name=f"season_params_{i}")
            for i, (num_seasons, len_per_season) in enumerate(custom_seas)
        ]
        self.reshape_layers = [
            Reshape(target_shape=(feat_dim, num_seasons))
            for num_seasons, len_per_season in custom_seas
        ]

    def _get_season_indexes_over_seq(self, num_seasons, len_per_season):
        season_indexes = tf.range(num_seasons)[:, None] + tf.zeros(
            (num_seasons, len_per_season), dtype=tf.int32
        )
        season_indexes = tf.reshape(season_indexes, [-1])
        season_indexes = tf.tile(season_indexes, [self.seq_len // len_per_season + 1])[
            : self.seq_len
        ]
        return season_indexes

    def call(self, z):
        N = tf.shape(z)[0]
        ones_tensor = tf.ones(shape=[N, self.feat_dim, self.seq_len], dtype=tf.int32)

        all_seas_vals = []
        for i, (num_seasons, len_per_season) in enumerate(self.custom_seas):
            season_params = self.dense_layers[i](z)  # shape: (N, D * S)
            season_params = self.reshape_layers[i](season_params)  # shape: (N, D, S)

            season_indexes_over_time = self._get_season_indexes_over_seq(
                num_seasons, len_per_season
            )

            dim2_idxes = ones_tensor * tf.reshape(
                season_indexes_over_time, shape=(1, 1, -1)
            )  # shape: (N, D, T)
            season_vals = tf.gather(
                season_params, dim2_idxes, batch_dims=-1
            )  # shape (N, D, T)

            all_seas_vals.append(season_vals)

        all_seas_vals = K.stack(all_seas_vals, axis=-1)  # shape: (N, D, T, S)
        all_seas_vals = tf.reduce_sum(all_seas_vals, axis=-1)  # shape (N, D, T)
        all_seas_vals = tf.transpose(all_seas_vals, perm=[0, 2, 1])  # shape (N, T, D)
        return all_seas_vals

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.seq_len, self.feat_dim)


# ============================================================================
# TIMEVAE MODEL
# ============================================================================

class TimeVAE(BaseVariationalAutoencoder):
    model_name = "TimeVAE"

    def __init__(
        self,
        hidden_layer_sizes,
        trend_poly=0,
        custom_seas=None,
        use_residual_conn=True,
        **kwargs,
    ):
        """
        TimeVAE: Temporal Variational Autoencoder
        
        Args:
            hidden_layer_sizes: list of number of filters in convolutional layers
            trend_poly: integer for number of orders for trend component
            custom_seas: list of tuples of (num_seasons, len_per_season)
            use_residual_conn: boolean for residual connection
        """

        super(TimeVAE, self).__init__(**kwargs)

        if hidden_layer_sizes is None:
            hidden_layer_sizes = [50, 100, 200]

        self.hidden_layer_sizes = hidden_layer_sizes
        self.trend_poly = trend_poly
        self.custom_seas = custom_seas
        self.use_residual_conn = use_residual_conn
        self.encoder = self._get_encoder()
        self.decoder = self._get_decoder()
        self.compile(optimizer=Adam())

    def _get_encoder(self):
        encoder_inputs = Input(
            shape=(self.seq_len, self.feat_dim), name="encoder_input"
        )
        x = encoder_inputs
        for i, num_filters in enumerate(self.hidden_layer_sizes):
            x = Conv1D(
                filters=num_filters,
                kernel_size=3,
                strides=2,
                activation="relu",
                padding="same",
                name=f"enc_conv_{i}",
            )(x)

        x = Flatten(name="enc_flatten")(x)

        self.encoder_last_dense_dim = x.shape[-1]

        z_mean = Dense(self.latent_dim, name="z_mean")(x)
        z_log_var = Dense(self.latent_dim, name="z_log_var")(x)

        encoder_output = Sampling()([z_mean, z_log_var])
        self.encoder_output = encoder_output

        encoder = Model(
            encoder_inputs, [z_mean, z_log_var, encoder_output], name="encoder"
        )
        return encoder

    def _get_decoder(self):
        decoder_inputs = Input(shape=(self.latent_dim,), name="decoder_input")

        outputs = None
        outputs = self.level_model(decoder_inputs)
        
        # trend polynomials
        if self.trend_poly is not None and self.trend_poly > 0:
            trend_vals = TrendLayer(self.feat_dim, self.trend_poly, self.seq_len)(
                decoder_inputs
            )
            outputs = trend_vals if outputs is None else outputs + trend_vals

        # custom seasons
        if self.custom_seas is not None and len(self.custom_seas) > 0:
            cust_seas_vals = SeasonalLayer(
                feat_dim=self.feat_dim,
                seq_len=self.seq_len,
                custom_seas=self.custom_seas,
            )(decoder_inputs)
            outputs = cust_seas_vals if outputs is None else outputs + cust_seas_vals

        if self.use_residual_conn:
            residuals = self._get_decoder_residual(decoder_inputs)
            outputs = residuals if outputs is None else outputs + residuals

        if outputs is None:
            raise ValueError(
                "Error: No decoder model to use. "
                "You must use one or more of: "
                "trend, custom seasonality, and/or residual connection."
            )

        decoder = Model(decoder_inputs, [outputs], name="decoder")
        return decoder

    def level_model(self, z):
        level_params = Dense(self.feat_dim, name="level_params", activation="relu")(z)
        level_params = Dense(self.feat_dim, name="level_params2")(level_params)
        level_params = Reshape(target_shape=(1, self.feat_dim))(
            level_params
        )  # shape: (N, 1, D)

        ones_tensor = tf.ones(
            shape=[1, self.seq_len, 1], dtype=tf.float32
        )  # shape: (1, T, 1)

        level_vals = level_params * ones_tensor
        return level_vals

    def _get_decoder_residual(self, x):
        x = Dense(self.encoder_last_dense_dim, name="dec_dense", activation="relu")(x)
        x = Reshape(target_shape=(-1, self.hidden_layer_sizes[-1]), name="dec_reshape")(
            x
        )

        for i, num_filters in enumerate(reversed(self.hidden_layer_sizes[:-1])):
            x = Conv1DTranspose(
                filters=num_filters,
                kernel_size=3,
                strides=2,
                padding="same",
                activation="relu",
                name=f"dec_deconv_{i}",
            )(x)

        # last de-convolution
        x = Conv1DTranspose(
            filters=self.feat_dim,
            kernel_size=3,
            strides=2,
            padding="same",
            activation="relu",
            name=f"dec_deconv_final",
        )(x)

        x = Flatten(name="dec_flatten")(x)
        x = Dense(self.seq_len * self.feat_dim, name="decoder_dense_final")(x)
        residuals = Reshape(target_shape=(self.seq_len, self.feat_dim))(x)
        return residuals

    def save(self, model_dir: str):
        os.makedirs(model_dir, exist_ok=True)
        super().save(model_dir)

        if self.custom_seas is not None:
            custom_seas_serializable = [
                (int(num_seasons), int(len_per_season))
                for num_seasons, len_per_season in self.custom_seas
            ]
        else:
            custom_seas_serializable = None

        dict_params = {
            "seq_len": self.seq_len,
            "feat_dim": self.feat_dim,
            "latent_dim": self.latent_dim,
            "reconstruction_wt": self.reconstruction_wt,
            "hidden_layer_sizes": list(self.hidden_layer_sizes),
            "trend_poly": self.trend_poly,
            "custom_seas": custom_seas_serializable,
            "use_residual_conn": self.use_residual_conn,
        }
        params_file = os.path.join(model_dir, f"{self.model_name}_parameters.pkl")
        joblib.dump(dict_params, params_file)

    @classmethod
    def load(cls, model_dir: str) -> "TimeVAE":
        params_file = os.path.join(model_dir, f"{cls.model_name}_parameters.pkl")
        dict_params = joblib.load(params_file)
        vae_model = TimeVAE(**dict_params)
        vae_model.load_weights(model_dir)
        vae_model.compile(optimizer=Adam())
        return vae_model

In [12]:
import numpy as np
import tensorflow as tf
import os
import time
import re
import random
import gc
from pathlib import Path

from Time_vae_model import TimeVAE 

# Set reproducibility seeds
np.random.seed(123)
random.seed(123)
tf.keras.utils.set_random_seed(123)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = '123'

def clear_memory():
    gc.collect()
    tf.keras.backend.clear_session()

def load_temporal_dataset(file_path: str):
    try:
        data = np.load(file_path, allow_pickle=True)
        return {
            'data': data['data'].astype(np.float32),
            'norm_params': data['norm_params'].item(),
            'feature_names': list(data['feature_names']),
            'index': data['index'],
            'metadata': data['metadata'].item()
        }
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def get_matching_datasets(base_dir: str, level: int, trial: int):
    base_name = f"level_{level}_trial_{trial}"
    return {
        'train': os.path.join(base_dir, "train", f"train_preprocessed_{base_name}.npz"),
        'val': os.path.join(base_dir, "val", f"val_preprocessed_{base_name}.npz"),
        'test': os.path.join(base_dir, "test", f"test_preprocessed_{base_name}.npz")
    }

def discover_available_trials(processed_dir: str, level: int):
    train_dir = os.path.join(processed_dir, "train")
    if not os.path.exists(train_dir):
        return []
    available_trials = []
    pattern = f"train_preprocessed_level_{level}_trial_*.npz"
    for file_path in Path(train_dir).glob(pattern):
        match = re.search(rf"level_{level}_trial_(\d+)", file_path.stem)
        if match:
            trial_num = int(match.group(1))
            dataset_paths = get_matching_datasets(processed_dir, level, trial_num)
            if all(os.path.exists(p) for p in dataset_paths.values()):
                available_trials.append(trial_num)
    return sorted(available_trials)

def validate_dataset_compatibility(train_data, val_data):
    issues = []
    if train_data['data'].shape[1:] != val_data['data'].shape[1:]:
        issues.append(f"Shape mismatch - Train: {train_data['data'].shape}, Val: {val_data['data'].shape}")
    if train_data['feature_names'] != val_data['feature_names']:
        issues.append("Feature name mismatch between train and val")
    if train_data['index'][-1] >= val_data['index'][0]:
        issues.append(f"Temporal overlap - Train ends: {train_data['index'][-1]}, Val starts: {val_data['index'][0]}")
    return issues

def save_training_history(history, save_path):
    history_dict = {}
    for key, values in history.history.items():
        history_dict[key] = np.array(values)
    np.savez_compressed(save_path, **history_dict)
    print(f"  ✓ Training history saved: {save_path}")

def generate_synthetic_data_original(vae_model, level, trial, ratio, generation_idx, save_dir,
                                     real_data, norm_params, norm_key, num_synthetic_samples=500):
    ratio_str = str(ratio).replace(".", "p")
    synthetic_dir = os.path.join(save_dir, "synthetic_data", f"ratio_{ratio_str}")
    os.makedirs(synthetic_dir, exist_ok=True)
    try:
        generation_seed = 123 + level * 1000 + trial * 100 + generation_idx * 10
        np.random.seed(generation_seed)
        tf.random.set_seed(generation_seed)
        print(f"  Generating {num_synthetic_samples} synthetic samples (Level {level}, Trial {trial}, Ratio {ratio}, seed: {generation_seed})...")
        synthetic_samples = vae_model.get_prior_samples(num_samples=num_synthetic_samples)
        synthetic_file = os.path.join(synthetic_dir, f"synthetic_original_level{level}_trial{trial}_ratio{ratio_str}.npz")
        np.savez_compressed(synthetic_file, synthetic_samples=synthetic_samples, norm_params=norm_params, norm_key=norm_key,
                            metadata={'num_samples': num_synthetic_samples, 'model_type': 'original_timeVAE', 'generation_method': 'prior_sampling',
                                      'level': level, 'trial': trial, 'ratio': ratio, 'generation_seed': generation_seed, 'num_features': synthetic_samples.shape[2]})
        print(f"  ✓ Synthetic data saved to {synthetic_file}")
        return synthetic_file
    except Exception as e:
        print(f"  ✗ Error generating synthetic data: {e}")
        return None

def train_original_timevae(processed_dir, save_dir, selected_level, selected_trials, selected_ratio,
                           norm_key='total_demand_clipped', generate_synthetic=True,
                           batch_size=10, rest_minutes=1):
    print("="*80)
    print("ORIGINAL TimeVAE Training Pipeline")
    print("="*80)
    total_start_time = time.time()
    train_dir = os.path.join(processed_dir, "train")
    if not os.path.exists(train_dir):
        print(f"Error: Train directory not found at {train_dir}")
        return
    os.makedirs(save_dir, exist_ok=True)
    analysis_dir = os.path.join(save_dir, "training_histories")
    models_dir = os.path.join(save_dir, "models")
    meta_dir = os.path.join(save_dir, "meta_histories")
    os.makedirs(analysis_dir, exist_ok=True); os.makedirs(models_dir, exist_ok=True); os.makedirs(meta_dir, exist_ok=True)
    all_results = []
    trial_batches = [selected_trials[i:i+batch_size] for i in range(0, len(selected_trials), batch_size)]
    num_batches = len(trial_batches)

    for batch_idx, trial_batch in enumerate(trial_batches, 1):
        print(f"\nBATCH {batch_idx}/{num_batches}: Trials {trial_batch[0]}-{trial_batch[-1]}")
        batch_start_time = time.time()
        for trial_in_batch, trial in enumerate(trial_batch, 1):
            print(f"\n[Batch {batch_idx}/{num_batches}] Trial {trial_in_batch}/{len(trial_batch)}: Trial {trial}")
            trial_start_time = time.time()
            trial_seed = 123 + selected_level * 1000 + trial * 100
            np.random.seed(trial_seed); tf.random.set_seed(trial_seed); random.seed(trial_seed)
            try:
                dataset_paths = get_matching_datasets(processed_dir, selected_level, trial)
                train_data = load_temporal_dataset(dataset_paths['train'])
                val_data = load_temporal_dataset(dataset_paths['val'])
                if train_data is None or val_data is None: continue
                train_x = train_data['data'].astype(np.float32)
                num_features = train_x.shape[2]; seq_len = train_x.shape[1]
                vae_model = TimeVAE(seq_len=seq_len, feat_dim=num_features, latent_dim=32, hidden_layer_sizes=[64, 128, 256],
                                    trend_poly=2, custom_seas=[(24, 1), (7, 24)], use_residual_conn=True, reconstruction_wt=3.0, batch_size=32)
                vae_model.fit_on_data(train_x, max_epochs=100, verbose=1)
                model_save_path = os.path.join(models_dir, f"original_timevae_level{selected_level}_trial{trial}")
                os.makedirs(model_save_path, exist_ok=True); vae_model.save(model_save_path)
                history_path = os.path.join(analysis_dir, f"history_original_level{selected_level}_trial{trial}.npz")
                class FakeHistory:
                    def __init__(self): self.history = {'loss': [], 'reconstruction_loss': [], 'kl_loss': []}
                save_training_history(FakeHistory(), history_path)
                meta_path = os.path.join(meta_dir, f"trial_meta_original_level{selected_level}_trial{trial}.npz")
                np.savez_compressed(meta_path, level=selected_level, trial=trial, seq_len=seq_len, num_features=num_features,
                                    latent_dim=32, training_seed=trial_seed, model_type='original_timeVAE')
                synthetic_files = {}
                if generate_synthetic:
                    num_synth = int(train_x.shape[0] * selected_ratio)
                    synthetic_file = generate_synthetic_data_original(vae_model, selected_level, trial, selected_ratio, 0, save_dir,
                                                                      train_x, train_data['norm_params'], norm_key, num_synthetic_samples=num_synth)
                    synthetic_files[selected_ratio] = synthetic_file
                all_results.append({'level': selected_level, 'trial': trial, 'time_minutes': (time.time() - trial_start_time)/60})
            except Exception as e:
                print(f"  ✗ Error in trial {trial}: {e}")
            finally: clear_memory()
        if batch_idx < num_batches:
            print(f"\n⏸️  Resting for {rest_minutes} minutes..."); time.sleep(rest_minutes * 60)

    summary_path = os.path.join(meta_dir, f"training_summary_original_level{selected_level}_ratio{str(selected_ratio).replace('.', 'p')}.npz")
    np.savez_compressed(summary_path, results=all_results, level=selected_level, ratio=selected_ratio)
    return all_results

if __name__ == "__main__":
    # Interactive Input
    print("\n" + "="*50)
    SELECTED_LEVEL = int(input("Enter aggregation level (e.g., 700): "))
    NUM_TRIALS = int(input("Enter number of trials to run (e.g., 1): "))
    print("="*50 + "\n")

    # Fixed paths/config
    PROCESSED_DIR = r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\cft-vae\data"
    SAVE_DIR = r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\Timevae\original_timevae_results"
    
    # Start timer
    overall_start_time = time.time()

    results = train_original_timevae(
        processed_dir=PROCESSED_DIR,
        save_dir=SAVE_DIR,
        selected_level=SELECTED_LEVEL,
        selected_trials=list(range(1, NUM_TRIALS + 1)),
        selected_ratio=1.0,
        norm_key='total_demand_clipped',
        generate_synthetic=True,
        batch_size=10,
        rest_minutes=1
    )

    # Calculate Total Simulation Time
    overall_end_time = time.time()
    total_seconds = overall_end_time - overall_start_time
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)

    print("\n" + "="*80)
    print("ALL SIMULATIONS COMPLETE")
    print(f"⏱️  Total Execution Time: {hours} hours and {minutes} minutes")
    print(f"📂 Results saved to: {SAVE_DIR}")
    print("="*80)

Enter aggregation level (e.g., 700):  700
Enter number of trials to run (e.g., 1):  1



ORIGINAL TimeVAE Training Pipeline

BATCH 1/1: Trials 1-1

[Batch 1/1] Trial 1/1: Trial 1
Epoch 1/100
20/20 [==============================] - 2s 14ms/step - loss: 58745.9332 - reconstruction_loss: 10651.8252 - kl_loss: 1194.7073 - lr: 0.0010
Epoch 2/100
20/20 [==============================] - 0s 14ms/step - loss: 10354.0133 - reconstruction_loss: 2394.7976 - kl_loss: 2390.2981 - lr: 0.0010
Epoch 3/100
20/20 [==============================] - 0s 14ms/step - loss: 8407.8934 - reconstruction_loss: 2124.4880 - kl_loss: 1756.1969 - lr: 0.0010
Epoch 4/100
20/20 [==============================] - 0s 14ms/step - loss: 7366.0013 - reconstruction_loss: 1760.8805 - kl_loss: 1870.4095 - lr: 0.0010
Epoch 5/100
20/20 [==============================] - 0s 14ms/step - loss: 6468.2672 - reconstruction_loss: 1471.5569 - kl_loss: 1889.1926 - lr: 0.0010
Epoch 6/100
20/20 [==============================] - 0s 14ms/step - loss: 5759.6391 - reconstruction_loss: 1224.5374 - kl_loss: 1886.3105 - lr: 0.0010


In [13]:
import pandas as pd
import numpy as np
import os
import re
import time
import warnings
import gc
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')


def clear_memory():
    """Force garbage collection and clear memory."""
    gc.collect()


def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error."""
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if len(df) == 0:
        return np.nan

    mask = np.abs(df["y_true"]) > 1e-10
    if mask.sum() == 0:
        return np.nan

    mape = np.mean(np.abs((df.loc[mask, "y_true"] - df.loc[mask, "y_pred"]) / df.loc[mask, "y_true"])) * 100
    return mape


def calculate_metrics(y_true, y_pred):
    """Calculate forecasting metrics (RMSE, MAE, R², MAPE)."""
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if len(df) == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'mape': np.nan, 'n': 0}

    rmse = np.sqrt(mean_squared_error(df["y_true"], df["y_pred"]))
    mae = mean_absolute_error(df["y_true"], df["y_pred"])
    r2 = r2_score(df["y_true"], df["y_pred"])
    mape = calculate_mape(df["y_true"], df["y_pred"])

    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape, 'n': len(df)}


def load_real_data(file_path):
    """Load real temporal data from NPZ file."""
    try:
        data = np.load(file_path, allow_pickle=True)
        return {
            'data': data['data'].astype(np.float32),
            'norm_params': data['norm_params'].item(),
            'feature_names': list(data['feature_names']),
            'index': data['index'],
            'metadata': data['metadata'].item()
        }
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None


def load_synthetic_data(file_path):
    """Load synthetic data from NPZ file."""
    try:
        data = np.load(file_path, allow_pickle=True)
        return data['synthetic_samples']
    except Exception as e:
        print(f"Error loading synthetic data {file_path}: {e}")
        return None


def denormalize_data(data, norm_params, feature_name='total_demand_clipped'):
    """Denormalize data using saved normalization parameters."""
    if norm_params is None or feature_name not in norm_params:
        return data

    norm_info = norm_params[feature_name]
    if 'min' in norm_info and 'max' in norm_info:
        return data * (norm_info['max'] - norm_info['min'] + 1e-7) + norm_info['min']
    return data


def create_features_and_targets(time_series_data):
    """Convert time series data into forecasting features and targets."""
    if len(time_series_data.shape) == 3:
        flattened = time_series_data[:, :, 0].flatten()
    else:
        flattened = time_series_data.flatten()

    time_index = pd.date_range(start='2020-01-01', periods=len(flattened), freq='H')
    df = pd.DataFrame({'demand': flattened}, index=time_index)

    features = pd.DataFrame(index=df.index)

    # Lag features
    for lag in [1, 2, 3, 24, 25, 48, 168]:
        features[f'lag_{lag}'] = df['demand'].shift(lag)

    # Rolling statistics
    features['roll_24h_mean'] = df['demand'].shift(1).rolling(24, min_periods=12).mean()
    features['roll_24h_std'] = df['demand'].shift(1).rolling(24, min_periods=12).std()
    features['roll_168h_mean'] = df['demand'].shift(1).rolling(168, min_periods=24).mean()

    # Time features
    features['hour'] = features.index.hour
    features['dayofweek'] = features.index.dayofweek
    features['month'] = features.index.month

    # Cyclical encoding
    features['hour_sin'] = np.sin(2 * np.pi * features.index.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * features.index.hour / 24)
    features['dayofweek_sin'] = np.sin(2 * np.pi * features.index.dayofweek / 7)
    features['dayofweek_cos'] = np.cos(2 * np.pi * features.index.dayofweek / 7)

    # Boolean features
    features['is_weekend'] = (features.index.dayofweek >= 5).astype(int)
    features['is_business_hour'] = (
        (features.index.hour >= 8) & 
        (features.index.hour <= 18) & 
        (features.index.dayofweek < 5)
    ).astype(int)

    combined = pd.concat([features, df['demand'].rename('target')], axis=1).dropna()
    X = combined.drop('target', axis=1)
    y = combined['target']

    return X, y


def discover_available_trials(processed_dir, level):
    """Discover available trials for a given level."""
    train_dir = os.path.join(processed_dir, "train")
    if not os.path.exists(train_dir):
        return []

    trials = []
    for file in Path(train_dir).glob(f"train_preprocessed_level_{level}_trial_*.npz"):
        match = re.search(r"trial_(\d+)", file.stem)
        if match:
            trial = int(match.group(1))
            test_file = os.path.join(processed_dir, "test", f"test_preprocessed_level_{level}_trial_{trial}.npz")
            if os.path.exists(test_file):
                trials.append(trial)

    return sorted(trials)


def get_synthetic_file_path(synthetic_dir, level, trial, ratio, model_type="conditional"):
    """Get path to synthetic file for given parameters.
    
    Args:
        model_type: 'conditional' (your model) or 'original' (their model)
    """
    ratio_map = {1: "1p0", 1.5: "1p5", 2: "2p0", 5: "5p0", 10: "10p0", 50: "50p0"}
    if ratio not in ratio_map:
        return None

    ratio_suffix = ratio_map[ratio]
    
    if model_type == "conditional":
        # Your model format
        file_path = os.path.join(
            synthetic_dir, f"ratio_{ratio_suffix}",
            f"synthetic_level{level}_trial{trial}_ratio{ratio_suffix}.npz"
        )
    elif model_type == "original":
        # Original model format
        file_path = os.path.join(
            synthetic_dir, f"ratio_{ratio_suffix}",
            f"synthetic_original_level{level}_trial{trial}_ratio{ratio_suffix}.npz"
        )
    else:
        return None

    return file_path if os.path.exists(file_path) else None


def initialize_models(random_state=42):
    """Initialize forecasting models."""
    return {
        'RandomForest': RandomForestRegressor(
            n_estimators=200, max_depth=10,
            random_state=random_state, n_jobs=-1
        ),
        'XGBoost': XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            random_state=random_state, n_jobs=-1
        ),
        'GradientBoosting': GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05,
            max_depth=5, random_state=random_state
        ),
        'SVM': SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
    }


def run_baseline_evaluation(X_train_real, y_train_real, X_test_real, y_test_real, models):
    """Train and evaluate baseline models on real data only."""
    baseline_results = {}
    
    for model_name, model in models.items():
        model_copy = model.__class__(**model.get_params())
        model_copy.fit(X_train_real, y_train_real)
        y_pred = model_copy.predict(X_test_real)
        baseline_results[model_name] = calculate_metrics(y_test_real, y_pred)

    return baseline_results


def run_synthetic_evaluation(X_synthetic_common, y_synthetic, X_test_common, y_test_real,
                            baseline_results, models, selected_level, trial, selected_ratio,
                            model_type):
    """Evaluate models trained on synthetic data only."""
    results = []

    for model_name, model in models.items():
        model_copy = model.__class__(**model.get_params())
        model_copy.fit(X_synthetic_common, y_synthetic)
        y_pred = model_copy.predict(X_test_common)

        synth_results = calculate_metrics(y_test_real, y_pred)
        baseline = baseline_results[model_name]

        rmse_improve = ((baseline['rmse'] - synth_results['rmse']) / baseline['rmse'] * 100) if baseline['rmse'] > 0 else 0
        r2_improve = ((synth_results['r2'] - baseline['r2']) / abs(baseline['r2']) * 100) if baseline['r2'] != 0 else 0
        mape_improve = ((baseline['mape'] - synth_results['mape']) / baseline['mape'] * 100) if baseline['mape'] > 0 else 0

        results.append({
            'level': selected_level, 'trial': trial, 'ratio': selected_ratio, 'model': model_name,
            'model_type': model_type,
            'training_type': 'synthetic_only',
            'baseline_rmse': baseline['rmse'], 'result_rmse': synth_results['rmse'], 'rmse_improvement_pct': rmse_improve,
            'baseline_r2': baseline['r2'], 'result_r2': synth_results['r2'], 'r2_improvement_pct': r2_improve,
            'baseline_mape': baseline['mape'], 'result_mape': synth_results['mape'], 'mape_improvement_pct': mape_improve,
            'synthetic_samples': len(X_synthetic_common)
        })

    return results


def run_augmented_evaluation(X_train_common, y_train_real, X_synthetic_common, y_synthetic,
                            X_test_common, y_test_real, baseline_results, models,
                            selected_level, trial, selected_ratio, model_type):
    """Evaluate models trained on augmented data (real + synthetic)."""
    results = []

    X_augmented = pd.concat([X_train_common, X_synthetic_common], ignore_index=True)
    y_augmented = pd.concat([y_train_real, y_synthetic], ignore_index=True)

    for model_name, model in models.items():
        model_copy = model.__class__(**model.get_params())
        model_copy.fit(X_augmented, y_augmented)
        y_pred = model_copy.predict(X_test_common)

        aug_results = calculate_metrics(y_test_real, y_pred)
        baseline = baseline_results[model_name]

        rmse_improve = ((baseline['rmse'] - aug_results['rmse']) / baseline['rmse'] * 100) if baseline['rmse'] > 0 else 0
        r2_improve = ((aug_results['r2'] - baseline['r2']) / abs(baseline['r2']) * 100) if baseline['r2'] != 0 else 0
        mape_improve = ((baseline['mape'] - aug_results['mape']) / baseline['mape'] * 100) if baseline['mape'] > 0 else 0

        results.append({
            'level': selected_level, 'trial': trial, 'ratio': selected_ratio, 'model': model_name,
            'model_type': model_type,
            'training_type': 'augmented',
            'baseline_rmse': baseline['rmse'], 'result_rmse': aug_results['rmse'], 'rmse_improvement_pct': rmse_improve,
            'baseline_r2': baseline['r2'], 'result_r2': aug_results['r2'], 'r2_improvement_pct': r2_improve,
            'baseline_mape': baseline['mape'], 'result_mape': aug_results['mape'], 'mape_improvement_pct': mape_improve,
            'real_samples': len(X_train_common), 'synthetic_samples': len(X_synthetic_common),
            'total_samples': len(X_augmented)
        })

    return results


def run_multi_synthetic_evaluation(processed_dir, synthetic_dirs, save_dir, selected_level,
                                   num_trials, selected_ratio, norm_key='total_demand_clipped',
                                   random_state=42, batch_size=10, rest_minutes=2):
    """
    Run forecasting evaluation comparing multiple synthetic datasets.
    
    Args:
        synthetic_dirs: Dict with keys 'conditional' and 'original' pointing to synthetic data folders
                       Example: {'conditional': '/path/to/your/synthetic', 
                                'original': '/path/to/their/synthetic'}
        selected_ratio: Synthetic ratio to compare
    """
    print("="*80)
    print("MULTI-SYNTHETIC FORECASTING EVALUATION")
    print("Comparing Your Model vs Original Model")
    print("="*80)
    print(f"Configuration:")
    print(f"  Level: {selected_level}")
    print(f"  Total Trials: {num_trials}")
    print(f"  Batch Size: {batch_size} trials")
    print(f"  Rest Period: {rest_minutes} minutes between batches")
    print(f"  Synthetic Ratio: {selected_ratio}")
    print(f"  Norm Key: {norm_key}")
    print(f"\nSynthetic Data Sources:")
    for model_type, path in synthetic_dirs.items():
        print(f"  {model_type}: {path}")
    print("="*80 + "\n")

    os.makedirs(save_dir, exist_ok=True)

    available_trials = discover_available_trials(processed_dir, selected_level)
    if not available_trials:
        print(f"Error: No trials found for level {selected_level}")
        return

    selected_trials = available_trials[:num_trials]
    print(f"Found {len(available_trials)} total trials")
    print(f"Will evaluate first {len(selected_trials)} trials\n")

    # Split trials into batches
    trial_batches = [
        selected_trials[i:i+batch_size] 
        for i in range(0, len(selected_trials), batch_size)
    ]
    
    num_batches = len(trial_batches)
    models = initialize_models(random_state)
    all_results = []
    total_start_time = time.time()

    for batch_idx, trial_batch in enumerate(trial_batches, 1):
        print("\n" + "="*80)
        print(f"BATCH {batch_idx}/{num_batches}: Trials {trial_batch[0]}-{trial_batch[-1]}")
        print("="*80 + "\n")
        
        batch_start_time = time.time()
        batch_results = []

        for trial_in_batch, trial in enumerate(trial_batch, 1):
            trial_start = time.time()
            print(f"[Batch {batch_idx}/{num_batches}] Trial {trial_in_batch}/{len(trial_batch)} (Trial {trial})")
            print("-" * 60)

            try:
                # Load real data
                train_path = os.path.join(
                    processed_dir, "train",
                    f"train_preprocessed_level_{selected_level}_trial_{trial}.npz"
                )
                test_path = os.path.join(
                    processed_dir, "test",
                    f"test_preprocessed_level_{selected_level}_trial_{trial}.npz"
                )

                train_data = load_real_data(train_path)
                test_data = load_real_data(test_path)

                if train_data is None or test_data is None:
                    print(f"  ✗ Skipping trial {trial} - could not load real data")
                    continue

                train_real = denormalize_data(train_data['data'][:, :, 0], train_data['norm_params'], norm_key)
                test_real = denormalize_data(test_data['data'][:, :, 0], test_data['norm_params'], norm_key)

                X_train_real, y_train_real = create_features_and_targets(train_real)
                X_test_real, y_test_real = create_features_and_targets(test_real)

                if len(X_train_real) == 0 or len(X_test_real) == 0:
                    print(f"  ✗ Skipping trial {trial} - could not create features")
                    continue

                # Get common features
                common_features = X_train_real.columns
                X_train_common = X_train_real[common_features]
                X_test_common = X_test_real[common_features]

                # Baseline evaluation (only once per trial)
                baseline_results = run_baseline_evaluation(X_train_common, y_train_real, X_test_common, y_test_real, models)

                # Evaluate both synthetic datasets
                for model_type, synthetic_dir in synthetic_dirs.items():
                    synthetic_file = get_synthetic_file_path(
                        synthetic_dir, selected_level, trial, selected_ratio, model_type
                    )
                    
                    if synthetic_file is None:
                        print(f"  ⚠️  {model_type} synthetic file not found (ratio {selected_ratio})")
                        continue

                    synthetic_data = load_synthetic_data(synthetic_file)
                    if synthetic_data is None:
                        print(f"  ⚠️  Could not load {model_type} synthetic data")
                        continue

                    X_synthetic, y_synthetic = create_features_and_targets(synthetic_data)
                    if len(X_synthetic) == 0:
                        print(f"  ⚠️  Could not create {model_type} synthetic features")
                        continue

                    X_synthetic_common = X_synthetic[common_features]

                    # Synthetic-only evaluation
                    synth_results = run_synthetic_evaluation(
                        X_synthetic_common, y_synthetic, X_test_common, y_test_real,
                        baseline_results, models, selected_level, trial, selected_ratio, model_type
                    )
                    batch_results.extend(synth_results)
                    all_results.extend(synth_results)

                    # Augmented evaluation
                    aug_results = run_augmented_evaluation(
                        X_train_common, y_train_real, X_synthetic_common, y_synthetic,
                        X_test_common, y_test_real, baseline_results, models,
                        selected_level, trial, selected_ratio, model_type
                    )
                    batch_results.extend(aug_results)
                    all_results.extend(aug_results)

                    print(f"  ✓ {model_type} evaluation complete")

                trial_elapsed = time.time() - trial_start
                print(f"  ✓ Trial {trial} completed in {trial_elapsed/60:.2f} minutes\n")

            except Exception as e:
                print(f"  ✗ Error processing trial {trial}: {e}")
                import traceback
                traceback.print_exc()
                continue
            finally:
                clear_memory()

        batch_time = (time.time() - batch_start_time) / 60
        print(f"\n✓ Batch {batch_idx} completed in {batch_time:.1f} minutes ({len(batch_results)} results)")

        # Rest between batches
        if batch_idx < num_batches:
            print(f"\n⏸️  Resting for {rest_minutes} minutes before next batch...")
            time.sleep(rest_minutes * 60)
            print("  ✓ Rest complete, resuming evaluation...\n")

    # Save and analyze results
    if all_results:
        results_df = pd.DataFrame(all_results)
        save_path = os.path.join(
            save_dir,
            f"comparison_level{selected_level}_ratio{str(selected_ratio).replace('.', 'p')}.csv"
        )
        results_df.to_csv(save_path, index=False)

        # Generate comparison summary
        print("\n" + "="*80)
        print("COMPARISON SUMMARY")
        print("="*80)
        
        for model_type in synthetic_dirs.keys():
            model_results = results_df[results_df['model_type'] == model_type]
            if len(model_results) > 0:
                print(f"\n{model_type.upper()} MODEL:")
                print(f"  Average RMSE: {model_results['result_rmse'].mean():.4f}")
                print(f"  Average R²: {model_results['result_r2'].mean():.4f}")
                print(f"  Average MAPE: {model_results['result_mape'].mean():.4f}")
                print(f"  Avg RMSE Improvement: {model_results['rmse_improvement_pct'].mean():.2f}%")

        total_time = (time.time() - total_start_time) / 60

        print(f"\n{'='*80}")
        print("EVALUATION COMPLETE")
        print("="*80)
        print(f"Total time: {total_time:.1f} minutes")
        print(f"Processed {len(selected_trials)} trials")
        print(f"Total results: {len(all_results)}")
        print(f"Results saved to: {save_path}")
        print("="*80)

        return results_df

    return None


if __name__ == "__main__":
    PROCESSED_DIR = r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\cft-vae\data"
    
    # BOTH synthetic directories
    SYNTHETIC_DIRS = {
        'conditional': r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\cft-vae\data\synthetic_data",
        'original': r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\Timevae\original_timevae_results\synthetic_data"
    }
    
    SAVE_DIR = r"C:\Users\bin150\OneDrive - UBC\Desktop\TIMEVAE-ubc\comparison_results"
    
    # Configuration
    SELECTED_LEVEL = 700
    NUM_TRIALS = 2 # Start small for testing
    SELECTED_RATIO = 1.0
    NORM_KEY = 'total_demand_clipped'
    RANDOM_STATE = 42
    
    BATCH_SIZE = 10
    REST_MINUTES = 1

    results_df = run_multi_synthetic_evaluation(
        processed_dir=PROCESSED_DIR,
        synthetic_dirs=SYNTHETIC_DIRS,
        save_dir=SAVE_DIR,
        selected_level=SELECTED_LEVEL,
        num_trials=NUM_TRIALS,
        selected_ratio=SELECTED_RATIO,
        norm_key=NORM_KEY,
        random_state=RANDOM_STATE,
        batch_size=BATCH_SIZE,
        rest_minutes=REST_MINUTES
    )

    if results_df is not None:
        print(f"\n✓ Comparison complete. Results saved to: {SAVE_DIR}")

MULTI-SYNTHETIC FORECASTING EVALUATION
Comparing Your Model vs Original Model
Configuration:
  Level: 700
  Total Trials: 2
  Batch Size: 10 trials
  Rest Period: 1 minutes between batches
  Synthetic Ratio: 1.0
  Norm Key: total_demand_clipped

Synthetic Data Sources:
  conditional: C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\cft-vae\data\synthetic_data
  original: C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\Timevae\original_timevae_results\synthetic_data

Found 100 total trials
Will evaluate first 2 trials


BATCH 1/1: Trials 1-2

[Batch 1/1] Trial 1/2 (Trial 1)
------------------------------------------------------------
  ✓ conditional evaluation complete
  ✓ original evaluation complete
  ✓ Trial 1 completed in 7.53 minutes

[Batch 1/1] Trial 2/2 (Trial 2)
------------------------------------------------------------
  ✓ conditional evaluation complete
  ✓ original evaluation complete
  ✓ Trial 2 completed in 8.04 minutes


✓ Batch 1 completed in 15.6 minutes 

In [20]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import os
from pathlib import Path
import warnings
import re
import time
import gc

warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Real data directory
PROCESSED_DIR = r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\cft-vae\data"
# Synthetic data directory (Original TimeVAE)
SYNTHETIC_DIR = r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\Timevae\original_timevae_results"
# Directory to save these specific forecasting results
SAVE_DIR      = r"C:\Users\bin150\OneDrive - UBC\Desktop\Publication\WR2\Timevae\original_forecasting_results"

RANDOM_STATE = 42
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================================================================
# METRICS & HELPERS
# ==============================================================================
def calculate_mape(y_true, y_pred):
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if len(df) == 0: return np.nan
    mask = np.abs(df["y_true"]) > 1e-10
    if mask.sum() == 0: return np.nan
    mape = np.mean(np.abs((df.loc[mask, "y_true"] - df.loc[mask, "y_pred"]) / df.loc[mask, "y_true"])) * 100
    return mape

def calculate_metrics(y_true, y_pred):
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if len(df) == 0:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'mape': np.nan, 'n': 0}
    
    rmse = np.sqrt(mean_squared_error(df["y_true"], df["y_pred"]))
    mae = mean_absolute_error(df["y_true"], df["y_pred"])
    r2 = r2_score(df["y_true"], df["y_pred"])
    mape = calculate_mape(df["y_true"], df["y_pred"])
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape, 'n': len(df)}

def load_real_data(file_path):
    try:
        data = np.load(file_path, allow_pickle=True)
        return {
            'data': data['data'].astype(np.float32),
            'norm_params': data['norm_params'].item(),
            'feature_names': list(data['feature_names']),
            'index': data['index'],
            'metadata': data['metadata'].item()
        }
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def load_synthetic_data(file_path):
    try:
        data = np.load(file_path, allow_pickle=True)
        return data['synthetic_samples']
    except Exception as e:
        print(f"Error loading synthetic data {file_path}: {e}")
        return None

def denormalize_data(data, norm_params, feature_name='total_demand_clipped'):
    if norm_params is None or feature_name not in norm_params:
        return data
    norm_info = norm_params[feature_name]
    if 'min' in norm_info and 'max' in norm_info:
        return data * (norm_info['max'] - norm_info['min'] + 1e-7) + norm_info['min']
    return data

def create_features_and_targets(time_series_data):
    if len(time_series_data.shape) == 3:
        flattened = time_series_data[:, :, 0].flatten()
    else:
        flattened = time_series_data.flatten()
    
    time_index = pd.date_range(start='2020-01-01', periods=len(flattened), freq='H')
    df = pd.DataFrame({'demand': flattened}, index=time_index)
    features = pd.DataFrame(index=df.index)
    
    for lag in [1, 2, 3, 24, 25, 48, 168]:
        features[f'lag_{lag}'] = df['demand'].shift(lag)
    
    features['roll_24h_mean'] = df['demand'].shift(1).rolling(24, min_periods=12).mean()
    features['roll_24h_std'] = df['demand'].shift(1).rolling(24, min_periods=12).std()
    features['roll_168h_mean'] = df['demand'].shift(1).rolling(168, min_periods=24).mean()
    
    features['hour'] = features.index.hour
    features['dayofweek'] = features.index.dayofweek
    features['month'] = features.index.month
    features['hour_sin'] = np.sin(2 * np.pi * features.index.hour / 24)
    features['hour_cos'] = np.cos(2 * np.pi * features.index.hour / 24)
    features['dayofweek_sin'] = np.sin(2 * np.pi * features.index.dayofweek / 7)
    features['dayofweek_cos'] = np.cos(2 * np.pi * features.index.dayofweek / 7)
    
    combined = pd.concat([features, df['demand'].rename('target')], axis=1).dropna()
    X = combined.drop('target', axis=1)
    y = combined['target']
    return X, y

# ==============================================================================
# DISCOVERY HELPERS
# ==============================================================================
def discover_available_levels():
    train_dir = os.path.join(PROCESSED_DIR, "train")
    if not os.path.exists(train_dir): return []
    train_files = list(Path(train_dir).glob("train_preprocessed_*.npz"))
    levels = sorted({int(re.search(r"level_(\d+)_trial_", f.stem).group(1)) for f in train_files})
    return levels

def discover_available_trials(level):
    train_dir = os.path.join(PROCESSED_DIR, "train")
    if not os.path.exists(train_dir): return []
    trials = []
    for file in Path(train_dir).glob(f"train_preprocessed_level_{level}_trial_*.npz"):
        match = re.search(r"trial_(\d+)", file.stem)
        if match:
            trial = int(match.group(1))
            test_file = os.path.join(PROCESSED_DIR, "test", f"test_preprocessed_level_{level}_trial_{trial}.npz")
            if os.path.exists(test_file):
                trials.append(trial)
    return sorted(trials)

def discover_available_ratios(level, trial):
    available_ratios = []
    ratio_map = {1: "1p0", 1.5: "1p5", 2: "2", 5: "5", 10: "10", 50: "50"}
    for ratio, ratio_suffix in ratio_map.items():
        file_path = os.path.join(SYNTHETIC_DIR, "synthetic_data", f"ratio_{ratio_suffix}", 
                                 f"synthetic_original_level{level}_trial{trial}_ratio{ratio_suffix}.npz")
        if os.path.exists(file_path):
            available_ratios.append(ratio)
    return sorted(available_ratios)

def get_synthetic_file_path(level, trial, ratio):
    ratio_map = {1: "1p0", 1.5: "1p5", 2: "2", 5: "5", 10: "10", 50: "50"}
    if ratio not in ratio_map: return None
    ratio_suffix = ratio_map[ratio]
    
    # Points specifically to 'original' TimeVAE synthetic files
    file_path = os.path.join(SYNTHETIC_DIR, "synthetic_data", f"ratio_{ratio_suffix}", 
                            f"synthetic_original_level{level}_trial{trial}_ratio{ratio_suffix}.npz")
    
    return file_path if os.path.exists(file_path) else None

# ==============================================================================
# MAIN EVALUATION LOGIC
# ==============================================================================
def run_forecasting_evaluation():
    # --- START TIMER ---
    overall_start_time = time.time()
    
    print("================================================================================")
    print("TIMEVAE FORECASTING EVALUATION (Baseline, Synthetic-Only, Augmented)")
    print("Targeting: Original TimeVAE Results")
    print("================================================================================")
    
    available_levels = discover_available_levels()
    if not available_levels: return

    print(f"\nAvailable Levels: {available_levels}")
    selected_level = int(input("Enter aggregation level: "))

    available_trials = discover_available_trials(selected_level)
    print(f"Available Trials ({len(available_trials)}): {available_trials}")
    num_trials = int(input(f"How many trials to process (1-{len(available_trials)}): "))
    selected_trials = available_trials[:num_trials]

    available_ratios = discover_available_ratios(selected_level, selected_trials[0])
    print(f"Available Ratios: {available_ratios}")
    selected_ratio = float(input("Enter synthetic ratio: "))

    BATCH_SIZE = 10
    REST_TIME = 15
    
    all_results = []
    models_dict = {
        'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE),
        'SVM': SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
    }
    
    trial_batches = [selected_trials[i:i + BATCH_SIZE] for i in range(0, len(selected_trials), BATCH_SIZE)]

    for batch_idx, batch_trials in enumerate(trial_batches):
        print(f"\n>>> PROCESSING BATCH {batch_idx + 1}/{len(trial_batches)} <<<")
        
        for trial in batch_trials:
            print(f"\n   --- Trial {trial} ---")
            train_path = os.path.join(PROCESSED_DIR, "train", f"train_preprocessed_level_{selected_level}_trial_{trial}.npz")
            test_path = os.path.join(PROCESSED_DIR, "test", f"test_preprocessed_level_{selected_level}_trial_{trial}.npz")
            syn_path = get_synthetic_file_path(selected_level, trial, selected_ratio)

            train_data = load_real_data(train_path)
            test_data = load_real_data(test_path)
            synthetic_raw = load_synthetic_data(syn_path)
            
            if not train_data or not test_data or synthetic_raw is None: 
                print(f"   [!] Missing files for trial {trial}. Skipping...")
                continue

            # Create Real Datasets
            train_real = denormalize_data(train_data['data'][:, :, 0], train_data['norm_params'])
            test_real = denormalize_data(test_data['data'][:, :, 0], test_data['norm_params'])
            X_train_real, y_train_real = create_features_and_targets(train_real)
            X_test_real, y_test_real = create_features_and_targets(test_real)

            # Create Synthetic Datasets
            X_syn, y_syn = create_features_and_targets(synthetic_raw)
            
            # Align features
            common_feats = X_train_real.columns.intersection(X_syn.columns)
            X_train_real, X_test_real, X_syn = X_train_real[common_feats], X_test_real[common_feats], X_syn[common_feats]

            # Augmented Dataset
            X_aug = pd.concat([X_train_real, X_syn], ignore_index=True)
            y_aug = pd.concat([y_train_real, y_syn], ignore_index=True)

            for name, model_template in models_dict.items():
                # --- 1. BASELINE ---
                m_base = model_template.__class__(**model_template.get_params())
                m_base.fit(X_train_real, y_train_real)
                res_base = calculate_metrics(y_test_real, m_base.predict(X_test_real))

                # --- 2. SYNTHETIC ONLY ---
                m_syn = model_template.__class__(**model_template.get_params())
                m_syn.fit(X_syn, y_syn)
                res_syn = calculate_metrics(y_test_real, m_syn.predict(X_test_real))

                # --- 3. AUGMENTED ---
                m_aug = model_template.__class__(**model_template.get_params())
                m_aug.fit(X_aug, y_aug)
                res_aug = calculate_metrics(y_test_real, m_aug.predict(X_test_real))

                # Calculate Improvement (Baseline vs Augmented)
                imp_pct = ((res_base['rmse'] - res_aug['rmse']) / res_base['rmse'] * 100) if res_base['rmse'] > 0 else 0
                print(f"      {name:18}: Base {res_base['rmse']:.3f} | Syn {res_syn['rmse']:.3f} | Aug {res_aug['rmse']:.3f} ({imp_pct:+.2f}%)")

                # Store for report
                for stype, metrics in [('baseline', res_base), ('synthetic_only', res_syn), ('augmented', res_aug)]:
                    all_results.append({
                        'level': selected_level, 'trial': trial, 'ratio': selected_ratio, 
                        'model': name, 'training_type': stype,
                        'baseline_rmse': res_base['rmse'], 
                        'result_rmse': metrics['rmse'], 
                        'rmse_improvement_pct': imp_pct if stype == 'augmented' else 0,
                        'result_r2': metrics['r2'], 'result_mape': metrics['mape']
                    })

        if batch_idx < len(trial_batches):
            gc.collect()
            time.sleep(REST_TIME)

    # Save results and print final benchmark
    if all_results:
        df_res = pd.DataFrame(all_results)
        
        # Define the specific filename and full path
        fname = f"results_level{selected_level}_ratio{str(selected_ratio).replace('.', 'p')}.csv"
        save_path = os.path.join(SAVE_DIR, fname)
        
        # Save to the CSV file
        df_res.to_csv(save_path, index=False)
        
        # --- TIMER CALCULATION ---
        overall_end_time = time.time()
        total_seconds = overall_end_time - overall_start_time
        hours = int(total_seconds // 3600)
        minutes = int((total_seconds % 3600) // 60)
        
        # Final Terminal Output
        print("\n" + "=" * 80)
        print("EVALUATION COMPLETE")
        print(f"⏱️  Total Execution Time: {hours} hours and {minutes} minutes")
        print(f"📂 Results saved to: {save_path}")
        print("=" * 80)
        
    else:
        print("\nNo results generated. Please check if your synthetic and real data paths are correct.")

    return all_results

if __name__ == "__main__":
    run_forecasting_evaluation()

TIMEVAE FORECASTING EVALUATION (Baseline, Synthetic-Only, Augmented)
Targeting: Original TimeVAE Results

Available Levels: [20, 70, 200, 700]


Enter aggregation level:  700


Available Trials (100): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


How many trials to process (1-100):  100


Available Ratios: [1]


Enter synthetic ratio:  1



>>> PROCESSING BATCH 1/10 <<<

   --- Trial 1 ---
      RandomForest      : Base 314.294 | Syn 1496.552 | Aug 316.375 (-0.66%)
      XGBoost           : Base 320.300 | Syn 1496.961 | Aug 323.800 (-1.09%)
      GradientBoosting  : Base 322.082 | Syn 1496.558 | Aug 311.751 (+3.21%)
      SVM               : Base 310.491 | Syn 1496.927 | Aug 311.803 (-0.42%)

   --- Trial 2 ---
      RandomForest      : Base 314.616 | Syn 1512.511 | Aug 314.129 (+0.15%)
      XGBoost           : Base 322.346 | Syn 1512.588 | Aug 318.365 (+1.24%)
      GradientBoosting  : Base 311.302 | Syn 1512.535 | Aug 314.085 (-0.89%)
      SVM               : Base 303.183 | Syn 1513.654 | Aug 302.806 (+0.12%)

   --- Trial 3 ---
      RandomForest      : Base 2229.097 | Syn 4607.270 | Aug 2278.541 (-2.22%)
      XGBoost           : Base 2607.442 | Syn 4607.421 | Aug 2678.482 (-2.72%)
      GradientBoosting  : Base 2252.409 | Syn 4607.302 | Aug 2542.455 (-12.88%)
      SVM               : Base 3711.736 | Syn 4607.480 